In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# TVP-VAR -> TVP-VMA -> GFEVD (stabilized version)
# ------------------------------------------------------------
# Input
#   - tvpvar_beta.npy
#   - tvpvar_cov.npy
#   - tvpvar_selected_lag.txt
#
# Output
#   - gfevd_last.csv
#   - gfevd_mean.csv
#   - gfevd_all.npy
#   - gfevd_diag_summary.csv
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

BETA_FILE = BASE_DIR / "tvpvar_beta.npy"
COV_FILE = BASE_DIR / "tvpvar_cov.npy"
LAG_FILE = BASE_DIR / "tvpvar_selected_lag.txt"

OUT_LAST = BASE_DIR / "gfevd_last.csv"
OUT_MEAN = BASE_DIR / "gfevd_mean.csv"
OUT_ALL = BASE_DIR / "gfevd_all.npy"
OUT_DIAG = BASE_DIR / "gfevd_diag_summary.csv"

H = 10

VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

EPS = 1e-12
RIDGE_EPS = 1e-8
STABILITY_TOL = 0.999

# -----------------------------
# Helpers
# -----------------------------
def make_psd(mat: np.ndarray, eps: float = RIDGE_EPS) -> np.ndarray:
    mat = 0.5 * (mat + mat.T)
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, eps, None)
    mat_psd = vecs @ np.diag(vals) @ vecs.T
    mat_psd = 0.5 * (mat_psd + mat_psd.T)
    mat_psd += np.eye(mat_psd.shape[0]) * eps
    return mat_psd


def load_inputs():
    beta_t = np.load(BETA_FILE)   # (T_eff, k, k*p)
    cov_t = np.load(COV_FILE)     # (T_eff, k, k)

    with open(LAG_FILE, "r", encoding="utf-8") as f:
        p = int(f.read().strip())

    if beta_t.ndim != 3:
        raise ValueError(f"beta_t shape 이상: {beta_t.shape}")
    if cov_t.ndim != 3:
        raise ValueError(f"cov_t shape 이상: {cov_t.shape}")

    T_eff, k, kp = beta_t.shape

    if p < 1:
        raise ValueError("lag는 1 이상이어야 합니다.")
    if kp != k * p:
        raise ValueError(f"beta_t 마지막 차원({kp}) != k*p({k*p})")

    return beta_t, cov_t, p, T_eff, k


def unpack_A_mats(beta_one_t: np.ndarray, p: int) -> list:
    k, _ = beta_one_t.shape
    A_list = []
    for lag in range(p):
        start = lag * k
        end = (lag + 1) * k
        A_list.append(beta_one_t[:, start:end])
    return A_list


def build_companion(A_list: list) -> np.ndarray:
    k = A_list[0].shape[0]
    p = len(A_list)

    if p == 1:
        return A_list[0].copy()

    top = np.hstack(A_list)
    bottom_left = np.eye(k * (p - 1))
    bottom_right = np.zeros((k * (p - 1), k))
    bottom = np.hstack([bottom_left, bottom_right])

    comp = np.vstack([top, bottom])
    return comp


def spectral_radius(A_list: list) -> float:
    comp = build_companion(A_list)
    vals = np.linalg.eigvals(comp)
    return float(np.max(np.abs(vals)))


def stabilize_A_list(A_list: list, tol: float = STABILITY_TOL) -> list:
    """
    불안정하면 전체 A를 동일 비율로 shrink.
    """
    rho = spectral_radius(A_list)
    if rho < tol:
        return A_list

    scale = tol / max(rho, EPS)
    return [A * scale for A in A_list]


def compute_phi_list(A_list: list, H: int) -> list:
    p = len(A_list)
    k = A_list[0].shape[0]

    phi = [np.eye(k)]
    for h in range(1, H):
        phi_h = np.zeros((k, k))
        for j in range(1, min(h, p) + 1):
            phi_h += A_list[j - 1] @ phi[h - j]
        phi.append(phi_h)

    return phi


def compute_gfevd_one_t(A_list: list, Sigma_t: np.ndarray, H: int) -> np.ndarray:
    Sigma_t = make_psd(Sigma_t)
    A_list = stabilize_A_list(A_list, tol=STABILITY_TOL)

    k = Sigma_t.shape[0]
    phi_list = compute_phi_list(A_list, H)

    gfevd = np.zeros((k, k), dtype=float)

    for i in range(k):
        e_i = np.zeros((k, 1))
        e_i[i, 0] = 1.0

        denom = 0.0
        for h in range(H):
            phi_h = phi_list[h]
            term = e_i.T @ phi_h @ Sigma_t @ phi_h.T @ e_i
            denom += float(term)

        denom = max(denom, EPS)

        for j in range(k):
            e_j = np.zeros((k, 1))
            e_j[j, 0] = 1.0

            sigma_jj = max(float(Sigma_t[j, j]), EPS)

            numer = 0.0
            for h in range(H):
                phi_h = phi_list[h]
                term = e_i.T @ phi_h @ Sigma_t @ e_j
                numer += float(term) ** 2

            gfevd[i, j] = (numer / sigma_jj) / denom

    gfevd = np.where(np.isfinite(gfevd), gfevd, 0.0)

    row_sums = gfevd.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums < EPS, 1.0, row_sums)
    gfevd = gfevd / row_sums

    return gfevd


# -----------------------------
# Main
# -----------------------------
def main():
    beta_t, cov_t, p, T_eff, k = load_inputs()

    if len(VAR_NAMES) != k:
        raise ValueError(f"VAR_NAMES 길이({len(VAR_NAMES)})와 k({k})가 다릅니다.")

    print("==========================================")
    print("TVP-VAR -> GFEVD (stabilized)")
    print("==========================================")
    print("T_eff :", T_eff)
    print("k     :", k)
    print("lag p :", p)
    print("H     :", H)
    print()

    gfevd_all = np.zeros((T_eff, k, k))
    diag_rows = []

    prev_good_gfevd = None

    for t in range(T_eff):
        A_list_raw = unpack_A_mats(beta_t[t], p)
        rho_raw = spectral_radius(A_list_raw)

        try:
            gfevd_t = compute_gfevd_one_t(A_list_raw, cov_t[t], H)

            if not np.all(np.isfinite(gfevd_t)):
                raise ValueError("non-finite gfevd detected")

            row_err = float(np.max(np.abs(gfevd_t.sum(axis=1) - 1.0)))
            diag_mean = float(np.mean(np.diag(gfevd_t)))
            offdiag_mean = float((gfevd_t.sum() - np.trace(gfevd_t)) / (k * (k - 1)))
            tci = float((gfevd_t.sum() - np.trace(gfevd_t)) / k * 100.0)

            prev_good_gfevd = gfevd_t

        except Exception:
            if prev_good_gfevd is not None:
                gfevd_t = prev_good_gfevd.copy()
                row_err = float(np.max(np.abs(gfevd_t.sum(axis=1) - 1.0)))
                diag_mean = float(np.mean(np.diag(gfevd_t)))
                offdiag_mean = float((gfevd_t.sum() - np.trace(gfevd_t)) / (k * (k - 1)))
                tci = float((gfevd_t.sum() - np.trace(gfevd_t)) / k * 100.0)
            else:
                gfevd_t = np.eye(k)
                row_err = 0.0
                diag_mean = 1.0
                offdiag_mean = 0.0
                tci = 0.0

        gfevd_all[t] = gfevd_t

        diag_rows.append({
            "t": t,
            "rho_raw": rho_raw,
            "row_err": row_err,
            "diag_mean": diag_mean,
            "offdiag_mean": offdiag_mean,
            "tci": tci,
        })

        if (t + 1) % 200 == 0 or (t + 1) == T_eff:
            print(f"processed: {t + 1}/{T_eff}")

    gfevd_last = gfevd_all[-1]
    gfevd_mean = gfevd_all.mean(axis=0)

    df_last = pd.DataFrame(gfevd_last, index=VAR_NAMES, columns=VAR_NAMES)
    df_mean = pd.DataFrame(gfevd_mean, index=VAR_NAMES, columns=VAR_NAMES)
    df_diag = pd.DataFrame(diag_rows)

    df_last.to_csv(OUT_LAST, encoding="utf-8-sig")
    df_mean.to_csv(OUT_MEAN, encoding="utf-8-sig")
    df_diag.to_csv(OUT_DIAG, index=False, encoding="utf-8-sig")
    np.save(OUT_ALL, gfevd_all)

    print()
    print("Saved files:")
    print(f" - {OUT_LAST}")
    print(f" - {OUT_MEAN}")
    print(f" - {OUT_ALL}")
    print(f" - {OUT_DIAG}")
    print()
    print("Last-period GFEVD:")
    print(df_last.round(4).to_string())
    print()
    print("Mean GFEVD:")
    print(df_mean.round(4).to_string())
    print()
    print("Diagnostic summary:")
    print(df_diag[["rho_raw", "row_err", "diag_mean", "offdiag_mean", "tci"]].describe().round(4).to_string())


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'tvpvar_selected_lag.txt'